# Evaluación por ejecución y análisis de errores

Proyecto 3 · Deep Learning y Sistemas Inteligentes · Sección 30

## Objetivo

Comparar baseline, QLoRA r=16, QLoRA r=32 y el experimento de reparación sobre los mismos 2 esquemas inéditos y 40 ejemplos.

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
assert (ROOT / "config.json").exists(), "Ejecuta el notebook desde la raíz o notebooks/."
ARTIFACTS = Path(os.environ.get("PROJECT3_ARTIFACTS", ROOT.parent / "artifacts_locales"))
ARTIFACTS.mkdir(parents=True, exist_ok=True)
SEED = 2026

## 1. Ejecutar las corridas

In [ ]:
RUN_EVALUATION = False  # Activar cuando los adapters existan.
commands = [
    [sys.executable, str(ROOT / 'scripts/run_validation.py'), '--name', 'baseline'],
    [sys.executable, str(ROOT / 'scripts/run_validation.py'), '--name', 'qlora_r16', '--adapter', str(ARTIFACTS / 'runs/qlora_r16')],
    [sys.executable, str(ROOT / 'scripts/run_validation.py'), '--name', 'qlora_r32', '--adapter', str(ARTIFACTS / 'runs/qlora_r32')],
]
if RUN_EVALUATION:
    for command in commands:
        subprocess.run(command, check=True)
commands

## 2. Comparar EX y elegir adapter

In [ ]:
runs = {}
for path in (ROOT / 'resultados/corridas').glob('*.json') if (ROOT / 'resultados/corridas').exists() else []:
    runs[path.stem] = json.loads(path.read_text(encoding='utf-8'))
runs

## 3. Experimento de inferencia

Después de elegir el ganador, ejecútalo otra vez con `--repair`. Esa bandera habilita una única reparación si MySQL devuelve error.

In [ ]:
# Ejemplo:
# Sustituye qlora_rXX por el ganador bajo ARTIFACTS / 'runs'.
# Ejecuta una corrida directa y otra con --repair usando scripts/run_validation.py.

## 4. Selección y análisis

Selecciona la mayor EX ponderada, copia con `scripts/select_adapter.py`, completa categorías de error y actualiza `resultados/resumen.json`. Conserva también resultados negativos.